# HISTORICAL LSTM walkthrough — superseded protocol

All saved outputs in this notebook belong to the old experiment, not the frozen
October 1–23 protocol. Do not re-execute these archival cells against the changed
source or quote their scores as current results. See `START_HERE.md` for the
approved initial train/validation/final-fit dates and causal inference modes.
After genuine new training completes, `build_handoff.py --execute` replaces this
notebook using only `outputs/frozen_october_2026/` results. Binary slides remain
historical and are not changed by this annotation.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
if not (ROOT/'src/oil_lstm.py').exists():
    ROOT = ROOT/'m3_ml/lstm'
assert (ROOT/'src/oil_lstm.py').exists(), 'Open this notebook from the project folder.'
sys.path.insert(0, str(ROOT/'src'))
from oil_lstm import OilLSTM, read_data, windows, targets, boundary, origins_between
summary = json.loads((ROOT/'outputs/run_summary.json').read_text())
print('Data snapshot:', summary['first_date'], 'to', summary['last_date'])

## 1. Audit the supplied inputs

The dictionary defines which columns are available at each origin. `target_wti` is excluded even as a lag. Negative prices are retained, so we use dollar changes rather than logarithms.

In [ ]:
df, features = read_data(ROOT.parents[1]/'data/processed/wti_daily.csv', ROOT.parents[1]/'data/processed/features.csv')
print('Shape:', df.shape, '| predictors:', len(features))
print('Missing cells:', df.isna().sum().sum(), '| duplicate dates:', df.date.duplicated().sum())
display(df[['date','target_wti','wti_spot_last_published']].head())
display(df.loc[df.target_wti <= 0, ['date','target_wti']])
assert 'target_wti' not in features

## 2. Understand the origin and horizon

Origin t is before 00:00 New York time on row t. The 60-row sequence ends at t; output h predicts y[t+h-1]. Features on row t are already published according to the supplied dictionary. The h=1 target on row t is not yet known. A ten-observation buffer and complete-path purging protect split boundaries.

In [ ]:
cut = boundary(df, '2021-12-31', 10)
ii = origins_between(len(df), 60, 0, cut)
print('Last model-selection training origin:', df.date.iloc[ii[-1]].date())
print('Last model-selection training label:', df.date.iloc[cut].date())
assert (ii+59 <= cut).all()
t = int(ii[-1])
display(pd.DataFrame({'horizon':[1,5,20,60], 'target_date':[df.date.iloc[t+h-1] for h in [1,5,20,60]]}))
display(Image(filename=str(ROOT/'outputs/data_split.png')))

## 3. Network and training objective

The network learns 60 standardized dollar residuals from the published WTI anchor. Price forecasts add the anchor and undo horizon scaling. The last hidden state feeds dropout and a linear output layer. The full runnable training source is `src/oil_lstm.py`.

In [ ]:
import inspect
print(inspect.getsource(OilLSTM))
net = OilLSTM(32, 32, 0.2)
print('Trainable parameters:', sum(p.numel() for p in net.parameters()))
print('Selected settings:', summary['selected']['config'])
print('Final epochs:', summary['selected']['epochs'])

## 4. Validation selects the configuration

The test period is not used for architecture, epoch, or ridge-alpha selection. After selection, final models refit on eligible pre-test complete paths. Three fixed seeds are averaged; test-based seed selection is prohibited.

In [ ]:
display(pd.read_csv(ROOT/'outputs/validation_search.csv'))
display(pd.read_csv(ROOT/'outputs/ridge_validation.csv'))
display(Image(filename=str(ROOT/'outputs/validation_curves.png')))

## 5. Untouched test comparisons

MAE and RMSE are USD/barrel. Positive relative improvement means better than the available published-price persistence benchmark. Different horizons have different sample counts; the common-origin CSV supplies a fixed-origin comparison. Direction is relative to the published anchor, not daily close-to-close direction.

In [ ]:
metrics = pd.read_csv(ROOT/'outputs/test_metrics_key_horizons.csv')
display(metrics[metrics.model.isin(['LSTM','Persistence','Ridge'])][['model','horizon','n','mae','rmse','rmse_improvement_pct','direction_vs_published_pct']].round(3))
display(Image(filename=str(ROOT/'outputs/accuracy_by_horizon.png')))
display(Image(filename=str(ROOT/'outputs/forecast_vs_actual.png')))

## 6. Robustness, uncertainty and ablation

The 60-horizon improvement is concentrated in 2026/high-OVX origins. Bootstrap ranges include zero. This does not establish secure superiority. The one-feature ablation is one seed rather than the main three-seed ensemble, limiting feature-value interpretation.

In [ ]:
robust = pd.read_csv(ROOT/'outputs/robustness.csv')
display(robust[(robust.model=='LSTM') & (robust.horizon==60)][['group_type','group','n','rmse','rmse_improvement_pct']].round(3))
display(pd.read_csv(ROOT/'outputs/bootstrap_improvement.csv').round(3))
display(metrics[metrics.model.isin(['LSTM','LSTM_published_WTI_only'])][['model','horizon','mae','rmse']].round(3))

## 7. Last supplied origin and daily refresh

This is a historical origin before 6 October 2026, not today. The h=1 miss is retained. Future target dates are blank because an actual observation calendar is unavailable. The seed range is sensitivity to initialization, not a prediction interval.

After appending newly aligned feature rows, run `python src/oil_lstm.py --forecast-only`. A current target is not required; weights remain frozen.

In [ ]:
latest = pd.read_csv(ROOT/'outputs/latest_origin_forecast.csv')
display(latest[latest.horizon.isin([1,5,20,60])].round(3))
display(Image(filename=str(ROOT/'outputs/latest_forecast_path.png')))

## 8. Reproduction and tests

The following cell is deliberately off by default so reviewing this executed notebook does not overwrite the saved experiment. Enable it to run the full code into a new output folder. The shipped seven verification tests check economically relevant alignment and leakage invariants.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    import subprocess
    subprocess.run([sys.executable, str(ROOT/'src/oil_lstm.py'), '--output', str(ROOT/'outputs_reproduced')], check=True)
print((ROOT/'outputs/verification.log').read_text(encoding='utf-8', errors='replace'))

## 9. Presentation conclusion

LSTM is a valid modern sequence-model comparison, but the evidence is mixed: h=60 RMSE improves 8.1% against delayed published-price persistence, MAE barely changes, ridge is stronger, and gains vary by regime. Use the three main presentation slides and retain these caveats. Full speaking notes, teammate integration steps, data provenance and references are in the accompanying Markdown files and PDF.